# Sensitivity studies

<div style="
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 12px 16px;
    max-width: 500px;
    background-color: #f9f9f9;
">
  <strong>✈️⚡ Electric Aircraft Design</strong><br>
  <br>
  <strong>Author:</strong> Florent LUTZ<br>
  <strong>Affiliation:</strong> ISAE-SUPAERO / DCAS <br>
  <strong>Course:</strong> Electric Aircraft Design<br>
</div>

For this study, the use of the Kodiak 100 hybridisation case presented in *Lutz, Florent, et al. "Study of hybridisation scenarios for turboprop aircraft in the general aviation segment." 34th Congress of the International Council of the Aeronautical Sciences, Florence, Italy. 2024.* is proposed. Besides the reasons discussed in the introductory Notebook, the Kodiak 100 was selected as a test case because DAHER's recent involvement in the [TAGINE project](https://www.safran-group.com/pressroom/french-aerospace-consortium-launches-initiative-develop-hybrid-electric-propulsion-general-aviation-2025-06-18) which aims at integrating a hybrid powertrain on a General Aviation aircraft and their proposed use of the Kodiak 100 as a test platform. Additionally, the range of viability of electrified aircraft is predicted to be between 50 and 200 nm [1], which coincides very well with the actual use of the Kodiak 100 (see Figure 2 in Notebook [00_Introduction](00_Introduction.ipynb)).

Regarding the propulsive architecture; a parallel hybrid powertrain will be considered. The thermal branch will be identical to the original design while the electric branch will consists in a battery driven DC network. The architecture is presented in the next Figure.

<p align="center">
  <img src="./resources/hybrid_kodiak_100.png" width="900">
  <br>
  <em>Figure 3 : Propulsive architecture for the hybrid Kodiak 100</a></em>
</p>

To accelerate a potential EIS of the hybridised Kodiak and to reduce time to certification, a retrofit approach will be adopted. This means that no redesign of the airframe will be made, keeping the same as the original design. This will be modelled as a constant geometry and a constant airframe weight. Additionally, the payload will be adjusted in order to ensure that the MTOW of the hybridised Kodiak is equal to or below that of the reference aircraft. This means that if the weight of the powertrain gets big enough, the payload capabilities of the hybrid design will be reduced. For that the following equation will be used:

$PL_{des} = min(PL_{max}, MTOW_{ref}-OWE-m_{fuel,des})$

Because of the technological maturity of today's electric components, range and performance reduction, when compared to their conventional counterpart, might be required for a feasible design of a hybrid aircraft. However, to set commercially realistic targets, we will pick a range following the results illustrated in Figure 2 in Notebook [00_Introduction](00_Introduction.ipynb). Looking at Figure 2, two thirds of the flights have range of 100 nm or less. Nonetheless, a range of 200 nm will be selected for the case of the hybridised Kodiak. While this is above the most frequently flown range, for a mission of 100 nm and considering the cruise we selected for the design mission of the original Kodiak 100, only half would be flown in cruise conditions according to the reference aircraft's Pilot Operating Handbook (POH). This range will serve as target for the hybridised version of the Kodiak 100. For comparison, we will evaluate the performances of the original Kodiak 100 on that same mission considering its maximum payload of 1140 kg. The aircraft will be assumed to be loaded with just enough fuel to perform the mission. This off-design evaluation will be conducted using the FAST-OAD-CS23-HE which allows for on- and off-design performances evaluation. The same flight profile will be assumed, only the range and payload will change.

## Kodiak 100 off-design evaluation

The off-design evaluation of the Kodiak 100 will be done in the next cell. As this is an off-design evaluation the names of the quantity of interest will differ slightly from the variables names in Notebook [00_Introduction](00_Introduction.ipynb). Next cell will be prepared but you are expected to enter the data corresponding to the range and payload selected.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
Because of the intricacies of the FAST-OAD-CS23-HE framework and although the propulsive architecture will be the same, the name of the powertrain description file will be slightly different. Its content remains identical though.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The introductory Notebook needs to be run for this Notebook to work. Indeed results like the geometry, aerodynamics and some weights will be reused.
</div>

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it.
</div>

In [ ]:
%%capture

# Import useful packages
import pathlib
import shutil
import warnings
import fastoad.api as oad
from utils import rename_variables_for_payload_range

# Write the paths to the files of interest
DATA_FOLDER_PATH = "data"
RESULT_FOLDER_PATH = "results"

path_to_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_configuration_off_design.yml"
path_to_design_output_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_final_output.xml"
path_to_off_design_input_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_input_off_design.xml"
path_to_off_design_final_input_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_final_input_off_design.xml"

# Generate the final input file for the off-design computation, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    shutil.copy(path_to_design_output_file, path_to_off_design_input_file)
    rename_variables_for_payload_range(path_to_off_design_input_file)
    oad.generate_inputs(path_to_configuration_file, path_to_off_design_input_file, overwrite=True)

# This is where the value for the off-design mission should be inputted. The cell won't run until it is filled in.
off_design_mission_range = ...  # In nm
off_design_mission_payload = ... # In kg

off_design_mission_input_datafile = oad.DataFile(path_to_off_design_final_input_file)
off_design_mission_input_datafile["data:mission:operational:range"].value[0] = off_design_mission_range
off_design_mission_input_datafile["data:mission:operational:range"].units = "NM"
off_design_mission_input_datafile["data:mission:operational:payload:mass"].value[0] = off_design_mission_payload
off_design_mission_input_datafile["data:mission:operational:payload:mass"].units = "kg"
off_design_mission_input_datafile["data:mission:operational:payload:CG:x"].value[0] = 4.1
off_design_mission_input_datafile.save()

with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_off_design_process = oad.evaluate_problem(path_to_configuration_file, overwrite=True)

Just like in the introductory Notebook, we can now look at the results for the mass and fuel consumption. Most importantly, FAST-OAD-CS23-HE also computes the emission of polutant stemming from the combustion of the fuel for the mission and its production. In case the aircraft uses a battery, it also computes the emissions associated with the production of the electricity. More specifically, it computes the equivalent warming potential of these species expressed as a function of the warming potential of CO2. Finally, to account for the aircraft's ability to transport passengers over a certain distance (severly hindered by the use of battery), it also computes this amount for each kg of payload and km travelled (in the realms of LCA this would be the functionnal unit). These values will be used as metrics to compare the designs in future computations.

We can also use the tool presented in the previous Notebook to view the evolution of the aircraft's parameters during the flight. 

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
from tabulate import tabulate
from IPython.display import HTML, display
import openmdao.api as om
from fastga_he.gui.performances_viewer import PerformancesViewer

# Access the results with a datafile, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_datafile = oad.DataFile(kodiak_100_off_design_process.output_file_path)

# Print the converted results with the actual value
cell_list = [
    ["Parameter", "Unit", "Computed value"], 
    ["Off-design mission TOW", "kg", om.convert_units(kodiak_100_datafile["data:mission:operational:TOW"].value[0], kodiak_100_datafile["data:mission:operational:TOW"].units, "kg")],
    ["Off-design mission fuel", "kg", om.convert_units(kodiak_100_datafile["data:mission:operational:fuel"].value[0], kodiak_100_datafile["data:mission:operational:fuel"].units, "kg")],
    ["Off-design mission emissions", "gCO2eq", om.convert_units(kodiak_100_datafile["data:environmental_impact:operational:emissions"].value[0], kodiak_100_datafile["data:environmental_impact:operational:emissions"].units, "g")],
    ["Off-design mission emission factor", "gCO2eq/kgPAX/km", kodiak_100_datafile["data:environmental_impact:operational:emission_factor"].value[0]],
]

display(HTML(tabulate(cell_list, tablefmt="html")))

# Define the path to the files containing mission data
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_propulsion_data_off_design.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_mission_file_off_design.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Look at the turboshaft power required throughout the mission and put a screenshot of it in your report. On that picture, showcase the maximum power and the cruise power. What can you say about the difference between the two ? <br>    
For future comparison, write down the turboshaft's rated power.
</div>

## Hybrid Kodiak 100 (Hybridization scheme A)

The goal of this Notebook is to study the design sensitivity of a hybrid electric aircraft depending on a few select parameter. The first of which being the hybridization scheme. In the first hybridization scheme we will consider, we will assume a constant hybridization ratio, meaning at each step of the flight X% of the power will come from the thermal branch and 1-X% from the electric branch. Let's start by investigating the influence of the hyrbidization ratio on the aircraft's metrics.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it. Try different values of the hybridization ratio between 75% and 95% and mark down the results. They will be used in a future cell for post processing.
</div>

In [ ]:
path_to_hybrid_A_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_configuration.yml"
path_to_hybrid_A_input_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_input.xml"
path_to_hybrid_A_final_input_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_final_input.xml"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    oad.generate_inputs(path_to_hybrid_A_configuration_file, path_to_hybrid_A_input_file, overwrite=True)
    
# This is where the value for the hybridization ratio should be inputted. The cell won't run until it is filled in.
hybridization_ratio = 95.0  # In percent, i.e. 80.0, 90.0, ...

hybrid_A_input_datafile = oad.DataFile(path_to_hybrid_A_final_input_file)
hybrid_A_input_datafile["data:propulsion:he_power_train:planetary_gear:planetary_gear_1:power_split"].value[0] = hybridization_ratio
hybrid_A_input_datafile["data:propulsion:he_power_train:planetary_gear:planetary_gear_1:power_split"].units = "percent"
hybrid_A_input_datafile.save()

# Now we run the process
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    hybrid_kodiak_100_scheme_A_process = oad.evaluate_problem(path_to_hybrid_A_configuration_file, overwrite=True)

# Access the results with a datafile, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    hybrid_A_output_datafile = oad.DataFile(hybrid_kodiak_100_scheme_A_process.output_file_path)
    
# Look at the results
cell_list = [
    ["Parameter", "Unit", "Computed value"], 
    ["MTOW", "kg", om.convert_units(hybrid_A_output_datafile["data:weight:aircraft:MTOW"].value[0], hybrid_A_output_datafile["data:weight:aircraft:MTOW"].units, "kg")],
    ["OWE", hybrid_A_output_datafile["data:weight:aircraft:OWE"].units, hybrid_A_output_datafile["data:weight:aircraft:OWE"].value[0]],
    ["Payload", hybrid_A_output_datafile["data:weight:aircraft:payload"].units, hybrid_A_output_datafile["data:weight:aircraft:payload"].value[0]],
    ["Propulsive system weight", hybrid_A_output_datafile["data:propulsion:he_power_train:mass"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:mass"].value[0]],
    ["Battery weight", hybrid_A_output_datafile["data:propulsion:he_power_train:battery_pack:battery_pack_1:mass"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:battery_pack:battery_pack_1:mass"].value[0]],
    ["Turboshaft maximum power", hybrid_A_output_datafile["data:propulsion:he_power_train:turboshaft:turboshaft_1:power_rating"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:turboshaft:turboshaft_1:power_rating"].value[0]],
    ["Electric motor maximum power", hybrid_A_output_datafile["data:propulsion:he_power_train:PMSM:motor_1:shaft_power_rating"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:PMSM:motor_1:shaft_power_rating"].value[0]],
    ["Emissions from fuel", hybrid_A_output_datafile["data:environmental_impact:sizing:fuel_emissions"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:fuel_emissions"].value[0]],
    ["Emissions from electricity", hybrid_A_output_datafile["data:environmental_impact:sizing:energy_emissions"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:energy_emissions"].value[0]],
    ["Emissions factor", hybrid_A_output_datafile["data:environmental_impact:sizing:emission_factor"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:emission_factor"].value[0]],
]

print("\n")

display(HTML(tabulate(cell_list, tablefmt="html")))

print("\n")

# Define the path to the files containing mission data
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_propulsion_data.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_mission_file.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

While only a few outputs of the sizing process are highlighted in the previous cell, you can explicitly access all of them using the next cell. Be careful however, the next cell will only print out the results of the latest execution of the previous cell. If you want to change the hybridization ratio, re-run the previous cell to update the results in the next cell.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
oad.variable_viewer(hybrid_kodiak_100_scheme_A_process.output_file_path)

Use the cell below to illustrate the variation of the parameters you've noted down in your calculation as a function of the hybridization ratio.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it. You can also customize the figure for a clearer display.
</div>

In [ ]:
# Import useful packages
import plotly.graph_objects as go

x_data = [75, 80, 85, 90, 95] # [..., ..., ...]
y_data = [4.65, 3.69, 3.21, 2.87, 2.64] # [..., ..., ...]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_data, 
        y=y_data
    )
)
fig.update_layout(
    height=600,
    width=900,
)
fig.show()

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Plot and comment about the evolution of: the total emissions, the emissions from fuel, the emissions from the electricity, the battery mass, the payload mass and the emission factor. Conclude about this hybridization scheme.<br>
For future comparison, write down the variation of the turboshaft's and electric motor's rated power.
</div>

It is important to recall that for this hybridization scenario, a retrofit approach was used. Therefore the MTOW is limited and cannot diverge. An interesting exercise could be to approach this hybridization scheme with a design from scratch approach. The MTOW would be free to vary and could thus potentially diverge.

## Hybrid Kodiak 100 (Hybridization scheme B)

The second hybridization scheme we will study is a configuration which aims to nullify the emissions during low altitude phases, particularly at the beginning of climb and at the end of descent. The reason for this interest is detailed in [2]:
> An important aspect to be analysed is the emission of the take-off and landing (LTO) phase, which includes the taxiing, take-off and climb phase up to an altitude of 915 m. This aspect is of particular interest for the improvement of the so-called local air quality, i.e. the air quality in the surrounding areas of the airport

## Bibliography

[1] Langford, John S., and David K. Hall. "Electrified aircraft propulsion." The Bridge 50.2 (2020): 21-27.
[2] Abu Salem, K., and G. Palaia. "Environmental implications of hybrid-electric regional aircraft: emissions and climate change." 34th Congress of the International Council of the Aeronautical Sciences, ICAS 2024. International Council of the Aeronautical Sciences, 2024.